# Data Quality Report 

This notebook documents data quality issues for the 5 variables:

- `media_type`
- `profession`
- `specialty`
- `campaign_start_date`
- `campaign_end_date`

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATA_PATH = "../data/BreakThroughTech_training_data_1.xlsx"
df = pd.read_excel(DATA_PATH)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

ModuleNotFoundError: No module named 'pandas'

# Columns at a glance

In [ ]:
MY_COLS = ["media_type", "profession", "specialty", "campaign_start_date", "campaign_end_date"]

overview = pd.DataFrame({
    "dtype": df[MY_COLS].dtypes.astype(str),
    "n_null": df[MY_COLS].isnull().sum(),
    "pct_null": (df[MY_COLS].isnull().mean() * 100).round(1),
    "n_unique": df[MY_COLS].nunique(dropna=True),
})
overview

,dtype,n_null,pct_null,n_unique
media_type,str,80,19.2,2
profession,object,56,13.4,13
specialty,object,57,13.7,9
campaign_start_date,str,3,0.7,205
campaign_end_date,str,3,0.7,1


---
## 1. `media_type`

### Issue found
Missing values. There are only two valid categories present (`EDITORIAL_VIDEOS`, `SPONSORED_VIDEOS`) with no spelling/casing inconsistencies

In [ ]:
n = len(df)
vc = df["media_type"].value_counts(dropna=False)
pct = (vc / n * 100).round(1)
pd.DataFrame({"count": vc, "pct_of_dataset": pct})

,count,pct_of_dataset
media_type,,
EDITORIAL_VIDEOS,243,58.3
SPONSORED_VIDEOS,94,22.5
NaN,80,19.2


### Rows/columns affected
- **1 column** (`media_type`)
- **80 / 417 rows (19.2%)** have a null `media_type`

### Is the missingness random?
Most missing `media_type` coincide with rows that are also missing `profession`, `specialty`, `country`, and `pillar`, suggesting a batch of records from a source/import that never captured audience-targeting metadata at all.

In [ ]:
missing_media = df[df["media_type"].isnull()]
print(f"Rows missing media_type: {len(missing_media)}")
print("\nOf those, how many are ALSO missing these other fields:")
print(missing_media[["profession", "specialty", "country", "pillar"]].isnull().sum())

Rows missing media_type: 80

Of those, how many are ALSO missing these other fields:
profession    56
specialty     56
country       56
pillar        56
dtype: int64


**Finding:** 56 of the 80 rows missing `media_type` are *also* missing `profession`, `specialty`, `country`, and `pillar` simultaneously. The remaining 24 rows are missing `media_type` only.

### Proposed fix
- Recode nulls to `"UNKNOWN"`
- Explore removing the rows that are missing meta data or removing the columns entirely

---
## 2. `profession`

### Issues found
1. **Missing values.**
2. **Mixed representation:** the column mixes numeric codes and free-text labels for what appears to be the same underlying concept with no data dictionary in the repo to translate the codes

In [ ]:
vc = df["profession"].value_counts(dropna=False)
pct = (vc / n * 100).round(1)
pd.DataFrame({"count": vc, "pct_of_dataset": pct})

,count,pct_of_dataset
profession,,
8,162,38.8
99,107,25.7
NaN,56,13.4
psychologist,45,10.8
physician assistant,16,3.8
19,6,1.4
20,5,1.2
11,4,1.0
physician,4,1.0


In [ ]:
types = df["profession"].dropna().apply(lambda x: type(x).__name__).value_counts()
print("Value-type breakdown of non-null `profession` entries:")
print(types)
print(f"\n-> {types.get('int', 0)} rows store profession as a numeric code")
print(f"-> {types.get('str', 0)} rows store profession as free text")

Value-type breakdown of non-null `profession` entries:
profession
int    296
str     65
Name: count, dtype: int64

-> 296 rows store profession as a numeric code
-> 65 rows store profession as free text


### Rows/columns affected
- **1 column** (`profession`)
- **56 / 417 rows (13.4%)** are null
- **296 / 417 rows (71.0%)** use a numeric code (`8`, `99`, `19`, `20`, `11`, `15`, `10`, `12`, `13`, `23`)
- **65 / 417 rows (15.6%)** use free text (`psychologist`, `physician assistant`, `physician`)
- **No codebook is present in this repo** mapping the numeric codes to a profession name

### Proposed fix
- **Request a data dictionary** from the Challenge Advisor mapping numeric profession codes (`8`, `99`, `10`–`23`) to human-readable categories before attempting to merge them with the text labels.
- Until a codebook is available, treat the numeric codes as their **own categorical levels**, separate from the text labels
- Recode the 56 nulls to an explicit `"UNKNOWN"` category 
- When encoding, cast the whole column to `string` dtype first (so `8` becomes `"8"`, not a float/int) to avoid dtype-mixing bugs during one-hot encoding

---
## 3. `specialty`

### Issues found
1. **Missing values.**
2. **A lot of undefined** — the string `"undefined"` fills most of the column 
3. **Mixed representation** — a small number of rows store a numeric code instead of text

In [ ]:
vc = df["specialty"].value_counts(dropna=False)
pct = (vc / n * 100).round(1)
pd.DataFrame({"count": vc, "pct_of_dataset": pct})

,count,pct_of_dataset
specialty,,
undefined,295,70.7
NaN,57,13.7
urology,46,11.0
vascular medicine,8,1.9
77,3,0.7
35,2,0.5
45,2,0.5
59,2,0.5
70,1,0.2


In [ ]:
informative = df["specialty"].isin(["urology", "vascular medicine"]).sum()
print(f"Rows with a genuinely usable specialty value in this column: {informative} / {n} ({informative/n*100:.1f}%)")
not_usable = n - informative
print(f"Rows that are null, 'undefined', or a numeric code: {not_usable} / {n} ({not_usable/n*100:.1f}%)")

print("\nFor comparison, the separate `npi_specialty` column:")
print(f"  nulls: {df['npi_specialty'].isnull().sum()}")
print(f"  unique categories: {df['npi_specialty'].nunique()}")
print(df["npi_specialty"].value_counts())

Rows with a genuinely usable specialty value in this column: 54 / 417 (12.9%)
Rows that are null, 'undefined', or a numeric code: 363 / 417 (87.1%)

For comparison, the separate `npi_specialty` column:
  nulls: 0
  unique categories: 14
npi_specialty
Nephrology                  48
Cardiology                  40
Primary Care                36
Internal Medicine           35
Rheumatology                33
Endocrinology               31
Oncology                    31
Dermatology                 30
Hematology                  30
Gastroenterology            28
Pulmonology                 27
Neurology                   24
Urology                     23
3320 - Anesthesiologists     1
Name: count, dtype: int64


### Rows/columns affected
- **1 column** (`specialty`)
- **57 / 417 rows (13.7%)** are null
- **295 / 417 rows (70.7%)** are the placeholder `"undefined"`
- **11 / 417 rows (2.6%)** store an uninterpretable numeric code
- Net: only **54 / 417 rows (12.9%)** (`urology`, `vascular medicine`) carry usable text in this column — **87.1% of the column is not directly usable as given**

### Proposed fix
- Recode `"undefined"` and the 11 numeric-code rows to `NaN` **before** any imputation — treating `"undefined"` as a real category would let a non-informative placeholder leak into the model as if it were meaningful signal.
- **Recommend using `npi_specialty` as the primary specialty feature for modeling** instead of `specialty`, since it is complete and standardized. Remove column from model entirely

---
## 4. `campaign_start_date`

### Issue found
Missing values only. Formatting and range are otherwise clean.

In [ ]:
print(f"Nulls: {df['campaign_start_date'].isnull().sum()} / {n}")

parsed_start = pd.to_datetime(df["campaign_start_date"], errors="coerce", utc=True)
unparseable = parsed_start.isnull() & df["campaign_start_date"].notnull()
print(f"Non-null values that fail to parse as a timestamp: {unparseable.sum()}")
print(f"Date range: {parsed_start.min()}  to  {parsed_start.max()}")

Nulls: 3 / 417
Non-null values that fail to parse as a timestamp: 0
Date range: 2020-12-17 23:00:40+00:00  to  2026-04-17 14:58:22+00:00


### Rows/columns affected
- **1 column** (`campaign_start_date`)
- **3 / 417 rows (0.7%)** are null
- **0 rows** fail to parse; the format is consistent (`YYYY-MM-DD HH:MM:SS.ffffff UTC`) across all non-null values
- Range (2020-12-17 to 2026-04-17) is a plausible multi-year span for a historical campaign dataset 

### Proposed fix
- Drop the 3 rows with a null start date — too few to impute reliably
- Convert the column from string to a proper `datetime64` dtype (currently stored as text) ahead of feature engineering (campaign age in days, month/quarter/year extraction, etc.).

---
## 5. `campaign_end_date`

### Issues found
1. **Missing values** — same 3 rows as `campaign_start_date`.
2. **Sentinel/placeholder value dominates the entire column** — this is the most significant issue found among my assigned variables.

In [ ]:
vc = df["campaign_end_date"].value_counts(dropna=False)
pct = (vc / n * 100).round(2)
pd.DataFrame({"count": vc, "pct_of_dataset": pct})

,count,pct_of_dataset
campaign_end_date,,
2099-12-31 23:59:59.999999 UTC,414,99.28
NaN,3,0.72


In [ ]:
SENTINEL = "2099-12-31 23:59:59.999999 UTC"
is_sentinel = df["campaign_end_date"] == SENTINEL
is_null = df["campaign_end_date"].isnull()

print(f"Rows equal to sentinel value {SENTINEL!r}: {is_sentinel.sum()} / {n} ({is_sentinel.mean()*100:.1f}%)")
print(f"Rows null: {is_null.sum()} / {n} ({is_null.mean()*100:.1f}%)")
print(f"Rows with a genuine, non-sentinel, non-null end date: {n - is_sentinel.sum() - is_null.sum()}")

# Same 3 rows null in both start and end date?
both_null = df["campaign_start_date"].isnull() & df["campaign_end_date"].isnull()
print(f"\nRows null in BOTH start and end date: {both_null.sum()}")

Rows equal to sentinel value '2099-12-31 23:59:59.999999 UTC': 414 / 417 (99.3%)
Rows null: 3 / 417 (0.7%)
Rows with a genuine, non-sentinel, non-null end date: 0

Rows null in BOTH start and end date: 3


**Finding:** every single non-null value in `campaign_end_date` (414/417 rows) is the exact same timestamp, `2099-12-31 23:59:59.999999 UTC`. This is a placeholder the source system almost certainly writes to mean "no end date set" / "campaign still open," not a real end date. Combined with the 3 nulls, **zero rows (0/417) have a genuine, informative end date** — the column currently carries no real information at all. 

In [ ]:
parsed_end = pd.to_datetime(df["campaign_end_date"], errors="coerce", utc=True)
naive_duration_days = (parsed_end - parsed_start).dt.days
print("If we naively computed campaign duration = end_date - start_date:")
print(naive_duration_days.describe())

If we naively computed campaign duration = end_date - start_date:
count      414.000000
mean     27583.198068
std        342.481458
min      26921.000000
25%      27341.000000
50%      27546.000000
75%      27802.500000
max      28868.000000
dtype: float64


As shown above, a naive `end - start` duration feature would spuriously span tens of thousands of days for nearly every row — pure noise driven entirely by the sentinel date, not real campaign length.

### Rows/columns affected
- **1 column** (`campaign_end_date`)
- **3 / 417 rows (0.7%)** are null
- **414 / 417 rows (99.3%)** are the constant sentinel value
- **417 / 417 rows (100%)** — i.e., the *entire column* carries no usable end-date information

### Proposed fix
- **Do not use raw `campaign_end_date`** or any `end − start` duration derived from it as a model feature 
- Because literally 0 rows have a real end date, **recommend dropping the column entirely** for modeling purposes


---
## Summary table — for the team's combined Data Quality Report

In [ ]:
summary = pd.DataFrame([
    {
        "column": "media_type",
        "issue": "Missing values (systematic — tied to a 56-row block also missing profession/specialty/country/pillar)",
        "rows_affected": 80,
        "pct_of_dataset": round(80/n*100, 1),
        "proposed_fix": "Recode null -> 'UNKNOWN' category; consider excluding the 56-row block",
    },
    {
        "column": "profession",
        "issue": "Missing values",
        "rows_affected": 56,
        "pct_of_dataset": round(56/n*100, 1),
        "proposed_fix": "Recode null -> 'UNKNOWN' category (overlaps the same missing-metadata block).",
    },
    {
        "column": "profession",
        "issue": "Mixed representation: numeric codes (8,99,10-23) vs free text (physician, physician assistant, psychologist), no codebook available",
        "rows_affected": 361,
        "pct_of_dataset": round(361/n*100, 1),
        "proposed_fix": "Request data dictionary from Challenge Advisor; treat numeric codes as separate categorical levels; cast to string dtype before encoding.",
    },
    {
        "column": "specialty",
        "issue": "Missing values",
        "rows_affected": 57,
        "pct_of_dataset": round(57/n*100, 1),
        "proposed_fix": "Recode to NaN, then impute/flag as 'UNKNOWN'.",
    },
    {
        "column": "specialty",
        "issue": "Placeholder 'undefined' acting as hidden null",
        "rows_affected": 295,
        "pct_of_dataset": round(295/n*100, 1),
        "proposed_fix": "Recode 'undefined' -> NaN before imputation or consider removing column entirely; use npi_specialty as primary specialty feature instead.",
    },
    {
        "column": "specialty",
        "issue": "Numeric codes that don't map consistently to npi_specialty",
        "rows_affected": 11,
        "pct_of_dataset": round(11/n*100, 1),
        "proposed_fix": "Recode -> NaN",
    },
    {
        "column": "campaign_start_date",
        "issue": "Missing values",
        "rows_affected": 3,
        "pct_of_dataset": round(3/n*100, 1),
        "proposed_fix": "Drop the 3 rows; convert column to datetime64 dtype for feature engineering.",
    },
    {
        "column": "campaign_end_date",
        "issue": "Constant sentinel value (2099-12-31) with zero genuine end dates in the entire column",
        "rows_affected": 414,
        "pct_of_dataset": round(414/n*100, 1),
        "proposed_fix": "Drop the column entirely for modeling; do not derive duration = end - start; flag to Challenge Advisor as a likely export bug.",
    },
])
summary

,column,issue,rows_affected,pct_of_dataset,proposed_fix
0,media_type,Missing values (systematic — tied to a 56-row ...,80,19.2,Recode null -> 'UNKNOWN' category; do not impu...
1,profession,Missing values,56,13.4,Recode null -> 'UNKNOWN' category (overlaps th...
2,profession,"Mixed representation: numeric codes (8,99,10-2...",361,86.6,Request data dictionary from Challenge Advisor...
3,specialty,Missing values,57,13.7,"Recode to NaN, then impute/flag as 'UNKNOWN'."
4,specialty,Placeholder 'undefined' acting as hidden null,295,70.7,Recode 'undefined' -> NaN before imputation; u...
5,specialty,Numeric codes that don't map consistently to n...,11,2.6,"Recode -> NaN; treat as data-entry artifacts, ..."
6,campaign_start_date,Missing values,3,0.7,Drop the 3 rows; convert column to datetime64 ...
7,campaign_end_date,Missing values,3,0.7,Drop with the same 3 rows dropped for campaign...
8,campaign_end_date,Constant sentinel value (2099-12-31) with zero...,414,99.3,Drop the column entirely for modeling; do not ...
